# Main cohort inference walkthrough

Runs `src.inference.Predictor` (the canonical `bert-base-uncased` encoder with
aspect + sentiment heads, see [README](../README.md#architecture-this-repo)) on
a handful of example reviews.

Uses a trained checkpoint if one exists at `models/checkpoint_best.pt`
(produced by `python main.py train`), otherwise falls back to the untrained
baseline — same behavior as `python main.py predict`.

In [1]:
import os
from pathlib import Path

# Notebooks live in notebooks/; every relative path in this repo
# (config.yaml, V2/config.yaml, models/, ...) assumes the process cwd
# is the repo root, same as running `python main.py ...` from the shell.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("cwd:", Path.cwd())

cwd: C:\GitHub\NLPTransformerAnalysis


In [2]:
from src.inference import Predictor, format_prediction

CHECKPOINT = "models/checkpoint_best.pt"

from pathlib import Path
if Path(CHECKPOINT).exists():
    predictor = Predictor.from_checkpoint(CHECKPOINT, config_path="config.yaml")
    print(f"Loaded trained checkpoint: {CHECKPOINT}")
else:
    predictor = Predictor.from_pretrained(config_path="config.yaml")
    print("No checkpoint found — using the untrained baseline (random head weights).")
    print("Run `python main.py train` first for meaningful predictions.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No checkpoint found — using the untrained baseline (random head weights).
Run `python main.py train` first for meaningful predictions.


## Example reviews

Taken from the README quick-start plus a couple of mixed-signal cases.

In [3]:
example_reviews = [
    "Great quality but shipping took 3 weeks",
    "The food was excellent but the waiter was rude and slow.",
    "Absolutely love this product! Great quality and fast shipping.",
    "Terrible experience. Broke after one week. Customer service was unhelpful.",
    "Decent for the price. Nothing special but gets the job done.",
    "Package arrived damaged. Took 3 weeks to get here. Very disappointed.",
]

results = predictor.predict_batch(example_reviews)
for r in results:
    print(format_prediction(r, verbose=True))
    print("-" * 70)

Text:      Great quality but shipping took 3 weeks
Aspect:    shipping (confidence: 31.8%)
Sentiment: negative (confidence: 54.1%)
Latency:   239.4ms
  Aspect probs:    {'quality': 0.1749, 'usability': 0.1148, 'value': 0.0963, 'shipping': 0.3176, 'customer_service': 0.2964}
  Sentiment probs: {'negative': 0.5408, 'neutral': 0.1488, 'positive': 0.3105}
----------------------------------------------------------------------
Text:      The food was excellent but the waiter was rude and slow.
Aspect:    shipping (confidence: 33.7%)
Sentiment: negative (confidence: 56.7%)
Latency:   239.4ms
  Aspect probs:    {'quality': 0.1107, 'usability': 0.1585, 'value': 0.1271, 'shipping': 0.337, 'customer_service': 0.2667}
  Sentiment probs: {'negative': 0.5668, 'neutral': 0.1229, 'positive': 0.3103}
----------------------------------------------------------------------
Text:      Absolutely love this product! Great quality and fast shipping.
Aspect:    shipping (confidence: 43.8%)
Sentiment: negative 

## Try your own text

In [4]:
your_text = "Setup was confusing at first but customer support walked me through it quickly."

result = predictor.predict_one(your_text)
print(format_prediction(result, verbose=True))

Text:      Setup was confusing at first but customer support walked me through it quickly.
Aspect:    shipping (confidence: 45.1%)
Sentiment: negative (confidence: 52.9%)
Latency:   186.5ms
  Aspect probs:    {'quality': 0.0853, 'usability': 0.1168, 'value': 0.0968, 'shipping': 0.4515, 'customer_service': 0.2496}
  Sentiment probs: {'negative': 0.5286, 'neutral': 0.1521, 'positive': 0.3194}


## Next steps

- [02_data_training_pipeline.ipynb](02_data_training_pipeline.ipynb) — how the
  weak-label data pipeline and training loop work.
- [03_v2_v3_feature_demos.ipynb](03_v2_v3_feature_demos.ipynb) — sarcasm
  routing, quantum uncertainty, span extraction, and the V3 hybrid backbone.